# 🚀 Vibe Coding LLM Fine-Tuning Notebook (Qwen2.5-Coder-7B + Unsloth QLoRA)

This notebook fine-tunes `Qwen/Qwen2.5-Coder-7B-Instruct` on Google Colab's **Free T4 GPU** using **Unsloth** and **QLoRA 4-Bit Quantization**.

### 📌 Execution Checklist:
1. Ensure GPU is enabled: **Runtime -> Change runtime type -> Select T4 GPU**.
2. Click **Runtime -> Run All**.
3. Paste your Hugging Face API Token when prompted in Cell 7 to auto-upload model weights.

In [ ]:
# Cell 1: GPU Verification
!nvidia-smi

In [ ]:
# Cell 2: Automated Package Installation (Unsloth + Unsloth Zoo + PyTorch + TRL + PEFT)
!pip install unsloth unsloth_zoo
!pip install "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets huggingface_hub triton

In [ ]:
# Cell 3: Load Base Model Qwen2.5-Coder-7B in 4-Bit Precision
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None # Auto detect (Float16 for Tesla T4, Bfloat16 for Ampere+)
load_in_4bit = True # 4-bit quantization reduces VRAM from 16GB to ~5.5GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print("✅ Qwen2.5-Coder-7B-Instruct successfully loaded in 4-bit precision!")

In [ ]:
# Cell 4: Configure QLoRA Adapters for Fast Parameter-Efficient Training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank matrix dimension
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Reduces VRAM by 30%
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)
print("✅ QLoRA adapters injected into Self-Attention & MLP projection matrices!")

In [ ]:
# Cell 5: Load & Format Vibe Coding Dataset (ChatML Prompt Template)
from datasets import load_dataset
import json

# Download Vibe Coding Dataset from GitHub Repo
dataset_url = "https://raw.githubusercontent.com/shawaz03/LLM/main/data/vibe_coding_dataset.json"
!wget -O vibe_coding_dataset.json {dataset_url}

with open("vibe_coding_dataset.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

formatted_samples = []
for item in raw_data:
    text = f"<|im_start|>system\n{item['system']}<|im_end|>\n<|im_start|>user\n{item['instruction']}<|im_end|>\n<|im_start|>assistant\n{item['response']}<|im_end|>"
    formatted_samples.append({"text": text})

from datasets import Dataset
dataset = Dataset.from_list(formatted_samples)
print(f"✅ Loaded & formatted {len(dataset)} ChatML samples for training!")

In [ ]:
# Cell 6: Execute Unsloth Trainer
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can speed up training for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # ~20 mins on T4 GPU
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Starting QLoRA Fine-Tuning...")
trainer_stats = trainer.train()
print("🎉 Training Complete!")

In [ ]:
# Cell 7: Push Fine-Tuned Model Adapter to Hugging Face Model Hub
from huggingface_hub import notebook_login
print("🔑 Login to Hugging Face:")
notebook_login()

repo_id = "shawaz03/vibe-coder-7b"
print(f"Pushing LoRA adapter to Hugging Face Model Hub: '{repo_id}'...")
model.push_to_hub_merged(repo_id, tokenizer, save_method = "lora", token = True)
print(f"🎉 SUCCESS! Model weights published live at: https://huggingface.co/{repo_id}")